In [45]:
%%capture
import nltk
import pandas as pd
from sklearn.metrics import classification_report
from nltk.classify import NaiveBayesClassifier
import spacy
import sys
!pip install spacy==3.4.4 scispacy==0.5.3

In [3]:
Drug_lib_df = pd.read_csv("drugLibTest_NLP Final Project Dataset.csv")
Drug_lib_df.head()

,Unnamed: 0,urlDrugName,rating,effectiveness,sideEffects,condition,benefitsReview,sideEffectsReview,commentsReview
0,1366,biaxin,9,Considerably Effective,Mild Side Effects,sinus infection,The antibiotic may have destroyed bacteria cau...,"Some back pain, some nauseau.",Took the antibiotics for 14 days. Sinus infect...
1,3724,lamictal,9,Highly Effective,Mild Side Effects,bipolar disorder,Lamictal stabilized my serious mood swings. On...,"Drowsiness, a bit of mental numbness. If you t...",Severe mood swings between hypomania and depre...
2,3824,depakene,4,Moderately Effective,Severe Side Effects,bipolar disorder,Initial benefits were comparable to the brand ...,"Depakene has a very thin coating, which caused...",Depakote was prescribed to me by a Kaiser psyc...
3,969,sarafem,10,Highly Effective,No Side Effects,bi-polar / anxiety,It controlls my mood swings. It helps me think...,I didnt really notice any side effects.,This drug may not be for everyone but its wond...
4,696,accutane,10,Highly Effective,Mild Side Effects,nodular acne,Within one week of treatment superficial acne ...,Side effects included moderate to severe dry s...,Drug was taken in gelatin tablet at 0.5 mg per...


In [4]:
print("number of drug_lib transcripts: ", len(Drug_lib_df))

number of drug_lib transcripts:  1036


In [ ]:
%%capture
import sys
!{sys.executable} -m pip install spacy==3.4.4 scispacy==0.5.3

In [40]:
from nltk.corpus import stopwords  # Stopword removal
from nltk.tokenize import word_tokenize  # Tokenization
from nltk.stem import WordNetLemmatizer  # Lemmatization

In [41]:
#Check  rows with null values
Drug_lib_df.isnull().sum().sort_values(ascending = False)

,0
Unnamed: 0,0
urlDrugName,0
rating,0
effectiveness,0
sideEffects,0
condition,0
benefitsReview,0
sideEffectsReview,0
commentsReview,0
vader_compound,0


In [8]:
#Remove rows with nulls
Drug_lib_df = Drug_lib_df[Drug_lib_df['sideEffectsReview'].notna()]
Drug_lib_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1013 entries, 0 to 1035
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Unnamed: 0         1013 non-null   int64 
 1   urlDrugName        1013 non-null   object
 2   rating             1013 non-null   int64 
 3   effectiveness      1013 non-null   object
 4   sideEffects        1013 non-null   object
 5   condition          1013 non-null   object
 6   benefitsReview     1008 non-null   object
 7   sideEffectsReview  1013 non-null   object
 8   commentsReview     1012 non-null   object
dtypes: int64(2), object(7)
memory usage: 79.1+ KB


In [9]:
#Remove rows with nulls
Drug_lib_df = Drug_lib_df[Drug_lib_df['benefitsReview'].notna()]
Drug_lib_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1008 entries, 0 to 1035
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Unnamed: 0         1008 non-null   int64 
 1   urlDrugName        1008 non-null   object
 2   rating             1008 non-null   int64 
 3   effectiveness      1008 non-null   object
 4   sideEffects        1008 non-null   object
 5   condition          1008 non-null   object
 6   benefitsReview     1008 non-null   object
 7   sideEffectsReview  1008 non-null   object
 8   commentsReview     1007 non-null   object
dtypes: int64(2), object(7)
memory usage: 78.8+ KB


In [10]:
#Remove rows with nulls
Drug_lib_df = Drug_lib_df[Drug_lib_df['commentsReview'].notna()]
Drug_lib_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1007 entries, 0 to 1035
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Unnamed: 0         1007 non-null   int64 
 1   urlDrugName        1007 non-null   object
 2   rating             1007 non-null   int64 
 3   effectiveness      1007 non-null   object
 4   sideEffects        1007 non-null   object
 5   condition          1007 non-null   object
 6   benefitsReview     1007 non-null   object
 7   sideEffectsReview  1007 non-null   object
 8   commentsReview     1007 non-null   object
dtypes: int64(2), object(7)
memory usage: 78.7+ KB


## Convert into Lower Case

In [11]:
#Convert sideEffectsReview  into lowercase
Drug_lib_df['sideEffectsReview'] = Drug_lib_df['sideEffectsReview'].str.lower()
Drug_lib_df.head()

,Unnamed: 0,urlDrugName,rating,effectiveness,sideEffects,condition,benefitsReview,sideEffectsReview,commentsReview
0,1366,biaxin,9,Considerably Effective,Mild Side Effects,sinus infection,The antibiotic may have destroyed bacteria cau...,"some back pain, some nauseau.",Took the antibiotics for 14 days. Sinus infect...
1,3724,lamictal,9,Highly Effective,Mild Side Effects,bipolar disorder,Lamictal stabilized my serious mood swings. On...,"drowsiness, a bit of mental numbness. if you t...",Severe mood swings between hypomania and depre...
2,3824,depakene,4,Moderately Effective,Severe Side Effects,bipolar disorder,Initial benefits were comparable to the brand ...,"depakene has a very thin coating, which caused...",Depakote was prescribed to me by a Kaiser psyc...
3,969,sarafem,10,Highly Effective,No Side Effects,bi-polar / anxiety,It controlls my mood swings. It helps me think...,i didnt really notice any side effects.,This drug may not be for everyone but its wond...
4,696,accutane,10,Highly Effective,Mild Side Effects,nodular acne,Within one week of treatment superficial acne ...,side effects included moderate to severe dry s...,Drug was taken in gelatin tablet at 0.5 mg per...


In [12]:
#Convert benefitsReview into lowercase
Drug_lib_df['commentsReview'] = Drug_lib_df['commentsReview'].astype(str).str.lower()

Drug_lib_df.head()

,Unnamed: 0,urlDrugName,rating,effectiveness,sideEffects,condition,benefitsReview,sideEffectsReview,commentsReview
0,1366,biaxin,9,Considerably Effective,Mild Side Effects,sinus infection,The antibiotic may have destroyed bacteria cau...,"some back pain, some nauseau.",took the antibiotics for 14 days. sinus infect...
1,3724,lamictal,9,Highly Effective,Mild Side Effects,bipolar disorder,Lamictal stabilized my serious mood swings. On...,"drowsiness, a bit of mental numbness. if you t...",severe mood swings between hypomania and depre...
2,3824,depakene,4,Moderately Effective,Severe Side Effects,bipolar disorder,Initial benefits were comparable to the brand ...,"depakene has a very thin coating, which caused...",depakote was prescribed to me by a kaiser psyc...
3,969,sarafem,10,Highly Effective,No Side Effects,bi-polar / anxiety,It controlls my mood swings. It helps me think...,i didnt really notice any side effects.,this drug may not be for everyone but its wond...
4,696,accutane,10,Highly Effective,Mild Side Effects,nodular acne,Within one week of treatment superficial acne ...,side effects included moderate to severe dry s...,drug was taken in gelatin tablet at 0.5 mg per...


In [13]:
#Convert commentsReview into lowercase
Drug_lib_df['benefitsReview'] = Drug_lib_df['benefitsReview'].str.lower()
Drug_lib_df.head()

,Unnamed: 0,urlDrugName,rating,effectiveness,sideEffects,condition,benefitsReview,sideEffectsReview,commentsReview
0,1366,biaxin,9,Considerably Effective,Mild Side Effects,sinus infection,the antibiotic may have destroyed bacteria cau...,"some back pain, some nauseau.",took the antibiotics for 14 days. sinus infect...
1,3724,lamictal,9,Highly Effective,Mild Side Effects,bipolar disorder,lamictal stabilized my serious mood swings. on...,"drowsiness, a bit of mental numbness. if you t...",severe mood swings between hypomania and depre...
2,3824,depakene,4,Moderately Effective,Severe Side Effects,bipolar disorder,initial benefits were comparable to the brand ...,"depakene has a very thin coating, which caused...",depakote was prescribed to me by a kaiser psyc...
3,969,sarafem,10,Highly Effective,No Side Effects,bi-polar / anxiety,it controlls my mood swings. it helps me think...,i didnt really notice any side effects.,this drug may not be for everyone but its wond...
4,696,accutane,10,Highly Effective,Mild Side Effects,nodular acne,within one week of treatment superficial acne ...,side effects included moderate to severe dry s...,drug was taken in gelatin tablet at 0.5 mg per...


In [14]:
#Convert sideEffects into lowercase
Drug_lib_df['sideEffects'] = Drug_lib_df['sideEffects'].str.lower()
Drug_lib_df.head()

,Unnamed: 0,urlDrugName,rating,effectiveness,sideEffects,condition,benefitsReview,sideEffectsReview,commentsReview
0,1366,biaxin,9,Considerably Effective,mild side effects,sinus infection,the antibiotic may have destroyed bacteria cau...,"some back pain, some nauseau.",took the antibiotics for 14 days. sinus infect...
1,3724,lamictal,9,Highly Effective,mild side effects,bipolar disorder,lamictal stabilized my serious mood swings. on...,"drowsiness, a bit of mental numbness. if you t...",severe mood swings between hypomania and depre...
2,3824,depakene,4,Moderately Effective,severe side effects,bipolar disorder,initial benefits were comparable to the brand ...,"depakene has a very thin coating, which caused...",depakote was prescribed to me by a kaiser psyc...
3,969,sarafem,10,Highly Effective,no side effects,bi-polar / anxiety,it controlls my mood swings. it helps me think...,i didnt really notice any side effects.,this drug may not be for everyone but its wond...
4,696,accutane,10,Highly Effective,mild side effects,nodular acne,within one week of treatment superficial acne ...,side effects included moderate to severe dry s...,drug was taken in gelatin tablet at 0.5 mg per...


In [15]:
#Convert effectiveness into lowercase
Drug_lib_df['effectiveness'] = Drug_lib_df['effectiveness'].str.lower()
Drug_lib_df.head()

,Unnamed: 0,urlDrugName,rating,effectiveness,sideEffects,condition,benefitsReview,sideEffectsReview,commentsReview
0,1366,biaxin,9,considerably effective,mild side effects,sinus infection,the antibiotic may have destroyed bacteria cau...,"some back pain, some nauseau.",took the antibiotics for 14 days. sinus infect...
1,3724,lamictal,9,highly effective,mild side effects,bipolar disorder,lamictal stabilized my serious mood swings. on...,"drowsiness, a bit of mental numbness. if you t...",severe mood swings between hypomania and depre...
2,3824,depakene,4,moderately effective,severe side effects,bipolar disorder,initial benefits were comparable to the brand ...,"depakene has a very thin coating, which caused...",depakote was prescribed to me by a kaiser psyc...
3,969,sarafem,10,highly effective,no side effects,bi-polar / anxiety,it controlls my mood swings. it helps me think...,i didnt really notice any side effects.,this drug may not be for everyone but its wond...
4,696,accutane,10,highly effective,mild side effects,nodular acne,within one week of treatment superficial acne ...,side effects included moderate to severe dry s...,drug was taken in gelatin tablet at 0.5 mg per...


In [16]:
#Remove stop words

# Download the 'punkt_tab' resource before tokenization
nltk.download('punkt_tab') # Download the necessary data for punkt tokenizer

word_tokenized = nltk.word_tokenize(Drug_lib_df['sideEffectsReview'][0])
nltk.download('stopwords')
stopwords = nltk.corpus.stopwords.words('english')
word_tokenized = [word for word in word_tokenized if word not in stopwords]
print(word_tokenized)

['back', 'pain', ',', 'nauseau', '.']


In [17]:
sideEffectsReview = Drug_lib_df['sideEffectsReview']
sideEffects = Drug_lib_df ['sideEffects']
data_with_labels =list(zip(sideEffectsReview, sideEffects))
print ("data_with_labels[0]: ", data_with_labels[0])

data_with_labels[0]:  ('some back pain, some nauseau.', 'mild side effects')


In [18]:
urlDrugName = Drug_lib_df['urlDrugName']
condition = Drug_lib_df ['condition']
data_with_labels =list(zip(urlDrugName, condition))
print ("data_with_labels[0]: ", data_with_labels[0])

data_with_labels[0]:  ('biaxin', 'sinus infection')


#### Download Vader Lexicon

In [19]:
nltk.download('vader_lexicon')

True

 ## Applying VADER Sentiment Analysis on Comments, Benefits, and Side Effects Reviews

In [20]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon')

# Initialize VADER
sid = SentimentIntensityAnalyzer()

def label_sentiment(score):
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

columns_to_check = ['commentsReview', 'benefitsReview', 'sideEffectsReview']

for col in columns_to_check:
    print(f"\nSentiment analysis for: {col}")
    Drug_lib_df['vader_compound'] = Drug_lib_df[col].astype(str).apply(lambda x: sid.polarity_scores(x)['compound'])
    Drug_lib_df['vader_label'] = Drug_lib_df['vader_compound'].apply(label_sentiment)
    sentiment_counts = Drug_lib_df['vader_label'].value_counts(normalize=True) * 100
    print(sentiment_counts)


Sentiment analysis for: commentsReview
vader_label
positive    36.345581
negative    35.451837
neutral     28.202582
Name: proportion, dtype: float64

Sentiment analysis for: benefitsReview
vader_label
positive    46.772592
negative    38.828203
neutral     14.399206
Name: proportion, dtype: float64

Sentiment analysis for: sideEffectsReview
vader_label
negative    54.419067
neutral     25.521351
positive    20.059583
Name: proportion, dtype: float64


In [21]:
%%capture
!pip install transformers torch

In [22]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TextClassificationPipeline

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)





from transformers import pipeline

# Load the sentiment analysis pipeline with truncation enabled
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    truncation=True,
)

#  Create Text Classification Pipeline
health_sentiment = TextClassificationPipeline(model=model, tokenizer=tokenizer, return_all_scores=False)


reviews = Drug_lib_df['benefitsReview'].astype(str).tolist()
sentiment_results = sentiment_pipeline(reviews)

# Extract labels and scores
sentiment_labels = [result['label'] for result in sentiment_results]
sentiment_scores = [result['score'] for result in sentiment_results]



Drug_lib_df['bert_sentiment_label'] = sentiment_labels
Drug_lib_df['bert_sentiment_score'] = sentiment_scores


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [23]:
# Select a subset of reviews (first 50 for speed, expand later)
sample_reviews = Drug_lib_df['benefitsReview'].dropna().head().tolist()

# Run sentiment prediction
predicted_sentiment = health_sentiment(sample_reviews)

index_values = Drug_lib_df.head(len(predicted_sentiment)).index

# Add star label to DataFrame using index_values
Drug_lib_df.loc[index_values, 'bert_star_label'] = [res['label'] for res in predicted_sentiment]


In [24]:
def star_to_sentiment(label):
    stars = int(label.split()[0])
    if stars <= 2:
        return 'negative'
    elif stars == 3:
        return 'neutral'
    else:
        return 'positive'

Drug_lib_df.loc[index_values, 'bert_star_label'] = [res['label'] for res in predicted_sentiment]




In [25]:
Drug_lib_df[['benefitsReview', 'bert_star_label', 'bert_sentiment_score']].head()

,benefitsReview,bert_star_label,bert_sentiment_score
0,the antibiotic may have destroyed bacteria cau...,3 stars,0.431390
1,lamictal stabilized my serious mood swings. on...,5 stars,0.633112
2,initial benefits were comparable to the brand ...,4 stars,0.320671
3,it controlls my mood swings. it helps me think...,5 stars,0.763892
4,within one week of treatment superficial acne ...,1 star,0.584940


In [26]:
def get_bert_sentiment(text):
    return sentiment_pipeline(text)[0]['label']

# Apply the function to the dataset
Drug_lib_df['benefitsReview_raw'] = Drug_lib_df['benefitsReview'].apply(
    lambda x: get_bert_sentiment(str(x))
)

# Convert rating (1-5 stars) to positive, neutral, negative
def simplify(label):
    stars = int(label[0])  # Extract the first digit
    if stars >= 4:
        return 'positive'
    elif stars == 3:
        return 'neutral'
    else:
        return 'negative'

Drug_lib_df['benefitsReview_sentiment'] = Drug_lib_df['benefitsReview_raw'].apply(
    simplify
)

# Show results
Drug_lib_df[
    ['benefitsReview', 'benefitsReview_raw', 'benefitsReview_sentiment']
].head()

,benefitsReview,benefitsReview_raw,benefitsReview_sentiment
0,the antibiotic may have destroyed bacteria cau...,3 stars,neutral
1,lamictal stabilized my serious mood swings. on...,5 stars,positive
2,initial benefits were comparable to the brand ...,4 stars,positive
3,it controlls my mood swings. it helps me think...,5 stars,positive
4,within one week of treatment superficial acne ...,1 star,negative


In [27]:
def get_bert_sentiment(text):
    return sentiment_pipeline(text)[0]['label']

# Apply the function to the dataset
Drug_lib_df['commentsReview_raw'] = Drug_lib_df['commentsReview'].apply(
    lambda x: get_bert_sentiment(str(x))
)

# Convert rating (1-5 stars) to positive, neutral, negative
def simplify(label):
    stars = int(label[0])  # Extract the first digit
    if stars >= 4:
        return 'positive'
    elif stars == 3:
        return 'neutral'
    else:
        return 'negative'

Drug_lib_df['commentsReview_sentiment'] = Drug_lib_df['commentsReview_raw'].apply(
    simplify
)

# Show results
Drug_lib_df[
    ['commentsReview', 'commentsReview_raw', 'commentsReview_sentiment']
].head()

,commentsReview,commentsReview_raw,commentsReview_sentiment
0,took the antibiotics for 14 days. sinus infect...,1 star,negative
1,severe mood swings between hypomania and depre...,2 stars,negative
2,depakote was prescribed to me by a kaiser psyc...,2 stars,negative
3,this drug may not be for everyone but its wond...,5 stars,positive
4,drug was taken in gelatin tablet at 0.5 mg per...,1 star,negative


## **Drug Effectiveness Classification**

In [46]:
%%capture
!pip install accelerate -U
%%capture
!pip install transformers
%%capture
!pip install evaluate
!pip install datasets
from transformers import Trainer, TrainingArguments, DistilBertForSequenceClassification, \
                         DistilBertTokenizerFast, DataCollatorWithPadding, pipeline
from datasets import Dataset
import evaluate
import pandas as pd
import numpy as np
import os
os.environ["WANDB_DISABLED"] = "true"
from sklearn.pipeline import Pipeline

# Build pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=3000)),
    ('clf', LogisticRegression(max_iter=300))
])

# Train model
pipeline.fit(X_train, y_train)

# Predict
y_pred = pipeline.predict(X_test)

# Evaluation
print("Classification Report:\n")
print(classification_report(y_test, y_pred))



## Import libraries

In [47]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Load data
Drug_lib_df = pd.read_csv("drugLibTest_NLP Final Project Dataset.csv")

# Map effectiveness to categories
def map_effectiveness(score):
    if score >= 8:
        return "High"
    elif score >= 5:
        return "Medium"
    else:
        return "Low"

Drug_lib_df['label'] = Drug_lib_df['rating'].apply(map_effectiveness)

# Handle missing values in 'benefitsReview' column before splitting
# Impute missing values with empty strings or drop rows with missing values
Drug_lib_df['benefitsReview'].fillna('', inplace=True)  # Impute with empty strings

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(Drug_lib_df['benefitsReview'], Drug_lib_df['label'], test_size=0.2, random_state=42)

# TF-IDF vectorization
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Logistic Regression
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_tfidf, y_train)
y_pred = clf.predict(X_test_tfidf)

# Classification Report
print("TF-IDF + Logistic Regression Results:")
print(classification_report(y_test, y_pred, target_names=["Low", "Medium", "High"]))

/tmp/ipykernel_2381/2423859604.py:23: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  Drug_lib_df['benefitsReview'].fillna('', inplace=True)  # Impute with empty strings


TF-IDF + Logistic Regression Results:
              precision    recall  f1-score   support

         Low       0.57      0.98      0.72       107
      Medium       0.81      0.25      0.38        53
        High       0.50      0.08      0.14        48

    accuracy                           0.59       208
   macro avg       0.63      0.44      0.41       208
weighted avg       0.62      0.59      0.50       208



In [48]:
#1. Imports
import os
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    pipeline
)


In [49]:
#2. Load and Prepare Data

Drug_lib_df = pd.read_csv("drugLibTest_NLP Final Project Dataset.csv")
# Use benefitReview as the text input and rating as the target
df = Drug_lib_df[['benefitsReview', 'rating']].dropna()

# Map ratings to categorical labels
def map_effectiveness(rating):
    if rating >= 8:
        return 2  # High
    elif rating >= 5:
        return 1  # Medium
    else:
        return 0  # Low

df['label'] = df['rating'].apply(map_effectiveness)


In [50]:
#3. Train/Test Split
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['benefitsReview'], df['label'], test_size=0.2, random_state=42
)


In [51]:
#4. Tokenization
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=128)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [52]:
#5. Dataset Formatting
class DrugReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.encodings['input_ids'][idx]),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx]),
            'labels': torch.tensor(self.labels[idx])
        }
    def __len__(self):
        return len(self.labels)

train_dataset = DrugReviewDataset(train_encodings, train_labels.tolist())
test_dataset = DrugReviewDataset(test_encodings, test_labels.tolist())


In [ ]:
#6. Training Setup (Disable wandb)
os.environ["WANDB_DISABLED"] = "true"

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=3
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)


In [ ]:
#7. Train the Model
trainer.train()


In [ ]:
#8. Evaluate the Model
results = trainer.evaluate()
print("Evaluation Results:", results)


In [ ]:
#9. Classification Report
pred_output = trainer.predict(test_dataset)
preds = np.argmax(pred_output.predictions, axis=1)

print("Classification Report:")
print(classification_report(test_labels, preds, target_names=["Low", "Medium", "High"]))
